# Rodrigo's Pristine Vector Diffusion - v0.1

The Disco guidance stack — CLIP ensemble, Dango cutouts, spherical distance, weighted
prompts, 1000-entry schedule strings — applied to a canvas of **closed cubic Bézier
paths** instead of a pixel tensor. Every iteration renders the vector scene with a
differentiable rasterizer (diffvg), pushes the CLIP gradient back through the
rasterizer into control points and fill colors, and steps Adam. The finished piece is
a true, editable, infinitely-scalable **SVG** — not a trace of a raster.

Optionally, a diffusion prior steers the paths too via Score Distillation Sampling:
either the classic **512×512 guided-diffusion UNet** (the same model the main
notebook samples with) or the fast **secondary model**.

Lineage: CLIPDraw (Frans et al. 2021, https://arxiv.org/abs/2106.14843) for
CLIP-through-rasterizer; VectorFusion (Jain et al., CVPR 2023,
https://arxiv.org/abs/2211.11319) for SDS-through-rasterizer and blob init;
diffvg (Li et al., SIGGRAPH Asia 2020, https://github.com/BachiLi/diffvg).

Sibling of `rodrigos-pristine-disco-diffusion.ipynb` — same repo clones, same models
directory, same settings-file convention (this one reads `settings-vector.json`).
There is no sampler here: the "diffusion" is an optimization loop, so `steps`
becomes `iterations`, and schedules index by fraction-of-run (no last-S-entries
quirk to compensate for — see DEVNOTES.md).


### Prompts

In [ ]:
text_prompts = {
    0: [
        "a lighthouse on a cliff at sunset, flat vector art, minimal poster:2",
        "waves crashing on rocks:1",
    ],
}


### Master Settings

In [ ]:
class VectorSettings:
    #@title Vector Settings

    #@markdown ---

    #@markdown #Basic Settings

    #@markdown 📘 `batch_name`: the name of your image batch

    #@markdown 📘 `set_seed`: 'random_seed' or an integer

    #@markdown 📘 `iterations`: Adam steps over the vector scene. 400 sketches, 700 is a solid default, 1500+ polishes

    #@markdown 📘 `width_height`: canvas in *coordinate* units. The SVG is resolution-independent — optimize small (384-512), export at any size. Larger canvases only slow the rasterizer

    #@markdown 📘 `path_schedule`: multi-scale progressive tiers `[start_fraction, count, radius_fraction]` — the vector analog of the raster overview→innercut arc. Large shapes block in the composition first; later tiers unlock mid-run, colored from the canvas under them, and carve detail (LIVE, CVPR 2022). Set to `None` to use a single flat tier of `num_paths`

    #@markdown 📘 tier placement is blue-noise (Mitchell best-candidate, the exact-count form of Poisson-disk): even coverage instead of clumpy uniform-random. With an `init_image`, tier 1 is weighted by the init's edge map and later tiers by |render − target| — detail spawns where the image needs it, still clump-free

    #@markdown 📘 `num_paths`: only used when `path_schedule` is None. 64 = minimal poster, 128 = balanced

    #@markdown 📘 `num_segments`: cubic segments per path. 4 keeps shapes simple and "vector-y"; more allows intricate silhouettes

    batch_name = "VectorDisco" #@param{type: 'string'}
    set_seed = 'random_seed' #@param{type: 'string'}
    iterations = 700 #@param{type: 'number'}
    width_height = [384, 384] #@param{type: 'raw'}
    path_schedule = [[0.0, 64, 0.22], [0.1, 96, 0.09], [0.18, 64, 0.04], [0.55, 128, 0.018]] #@param{type: 'raw'}
    num_paths = 128 #@param{type: 'number'}
    num_segments = 4 #@param{type: 'number'}

    #@markdown ---

    #@markdown #Guidance (same semantics as the main notebook)

    #@markdown 📘 cut schedules are 1000-entry strings; here they index by fraction of the run (entry `int(1000*i/iterations)`) — no window correction needed

    #@markdown 📘 `cutn_batches`: accumulate CLIP gradient from multiple batches of cuts

    cut_overview = "[12]*400+[4]*600" #@param{type: 'string'}
    cut_innercut = "[4]*400+[12]*600" #@param{type: 'string'}
    cut_ic_pow = "[1]*1000" #@param{type: 'string'}
    cut_icgray_p = "[0.2]*400+[0]*600" #@param{type: 'string'}
    cutn_batches = 1 #@param{type: 'number'}
    skip_augs = False #@param{type: 'boolean'}

    #@markdown ---

    #@markdown #CLIP ensemble — defaults mirror the raster notebook's classic four

    ViTB32 = True #@param{type: 'boolean'}
    ViTB16 = True #@param{type: 'boolean'}
    ViTL14 = True #@param{type: 'boolean'}
    RN50 = True #@param{type: 'boolean'}
    RN50x4 = False #@param{type: 'boolean'}

    #@markdown 🧠 modern OpenCLIP (fp16, as in the raster notebook) — heavier, semantically richer

    ViTL14_laion2b = False #@param{type: 'boolean'}
    ViTH14_laion2b = False #@param{type: 'boolean'}

    #@markdown 📘 `quality_spherical_mean`: Karcher mean of cutout embeddings (same switch as the raster notebook)

    quality_spherical_mean = False #@param{type: 'boolean'}

    #@markdown ---

    #@markdown #Score Distillation (optional diffusion prior)

    #@markdown 📘 `sds_mode`: 'none' = pure CLIP guidance (CLIPDraw-style; crisp flat shapes). 'secondary' = fast prior from the secondary model (+0.25s/it). 'primary' = the 512×512 guided-diffusion UNet itself (+0.4s/it). The unconditional priors pull toward photographic texture — they add coherence at low scale but fight flatness at high scale. Start at 'none' or 'secondary' with sds_scale 100-300

    #@markdown 📘 `sds_t_range`: fraction-of-noise interval the prior is queried in

    #@markdown 📘 `sds_anneal`: slide the prior's timestep window high→low over the run (DreamTime-style) — structure gets locked early, detail refined late. Overrides sds_t_range when on

    sds_mode = 'primary' #@param ['none', 'secondary', 'primary']
    sds_scale = 150 #@param{type: 'number'}
    sds_t_range = [0.05, 0.95] #@param{type: 'raw'}
    sds_anneal = True #@param{type: 'boolean'}

    #@markdown ---

    #@markdown #Raster warm-start (VectorFusion recipe — maximum cohesion)

    #@markdown 📘 `init_image`: path to an image (e.g. a render from the raster notebook). The vector scene first FITS this image (init_fit_frac of the run, plain image loss), inheriting its full composition, then CLIP+SDS stylize it. `init_scale` keeps a weighted anchor to the image afterwards (0 releases it completely)

    #@markdown 🧠 for maximum inheritance, schedule ALL path tiers inside the fit phase (e.g. tiers at 0/0.1/0.18 with init_fit_frac 0.25) so the full shape budget vectorizes the target before stylization

    init_image = '' #@param{type: 'string'}
    init_fit_frac = 0.25 #@param{type: 'number'}
    init_scale = 100 #@param{type: 'number'}

    #@markdown ---

    #@markdown #Quality rewards (2026)

    #@markdown 📘 `shape_reg_scale`: shape-regularity reward — isoperimetric compactness + turning-angle smoothness over the boundary. Kills sickle/"chaotic cutout" shapes; also acts as anti-self-intersection (cf. VectorFusion's Xing loss). 0 disables

    #@markdown 📘 `aesthetic_scale`: LAION aesthetic-predictor reward (MLP over the ViT-L/14 CLIP embedding, Schuhmann's improved-aesthetic-predictor) — ascend a learned "does this look good" score. Needs ViTL14 enabled

    #@markdown 📘 `palette_scale` + `palette_k`: learned-palette cohesion — K palette anchors (k-means-initialized from the warm-start target) optimized jointly; every fill is softly attracted to its nearest anchor. Screen-print color unity

    #@markdown 📘 `solidity_scale`: pushes opacities to the ceiling for clean flat layering

    #@markdown 📘 `gradient_tiers`: leading path tiers rendered with two-stop linear gradients (learned second color + axis per shape; SVG exports native linearGradients). Big background tiers with gradients + flat detail tiers on top mirrors how poster artists work — smooth skies/water without spending shapes. 0 = all flat

    gradient_tiers = 2 #@param{type: 'number'}
    shape_reg_scale = 0.3 #@param{type: 'number'}
    aesthetic_scale = 0.5 #@param{type: 'number'}
    palette_scale = 5 #@param{type: 'number'}
    palette_k = 8 #@param{type: 'number'}
    solidity_scale = 2 #@param{type: 'number'}

    #@markdown ---

    #@markdown #Optimizer

    #@markdown 📘 `points_lr` is in canvas units; `color_lr` in [0,1] RGBA units

    #@markdown 📘 `background`: canvas backdrop each iteration; 'random' (a new color per iteration, CLIPDraw-style) is available as an anti-transparency-cheating experiment

    points_lr = 1.0 #@param{type: 'number'}
    color_lr = 0.02 #@param{type: 'number'}
    background = 'auto' #@param ['auto', 'white', 'black', 'random']

    #@markdown ---

    #@markdown #Run Settings

    display_rate = 50 #@param{type: 'number'}
    intermediate_saves = 150 #@param{type: 'number'}
    export_png_size = 1024 #@param{type: 'number'}

    #@markdown 📘 `perf_compile`: torch.compile the CLIP visual towers + the SDS UNet (one-time warmup of ~1-2 min on first run, cached afterwards)

    #@markdown 📘 `perf_autocast`: fp16 autocast for the CLIP guidance block. Measured NEUTRAL on top of perf_compile (0.480 vs 0.484 s/it) — kept as an experimental switch, off by default

    perf_compile = True #@param{type: 'boolean'}
    perf_autocast = False #@param{type: 'boolean'}

    #@markdown 📘 `renderer`: 'auto' prefers Bézier Splatting (NeurIPS 2025) — a GPU-native splatting rasterizer, ~270x faster than diffvg-on-CPU (3.7ms vs ~1000ms fwd+bwd at 512x768/320 shapes, measured) — and falls back to diffvg when gsplat isn't available. Shapes are two-cubic-half closed Béziers either way; SVG export is native in both

    renderer = 'auto' #@param ['auto', 'splat', 'diffvg']

    #@markdown 📘 `render_gpu`: diffvg backend only — 'auto' uses its CPU rasterizer on WSL2 (diffvg's managed-memory GPU path is ~30x slower there — measured, see DEVNOTES) and the GPU elsewhere

    render_gpu = 'auto' #@param ['auto', 'on', 'off']

settings = VectorSettings()

import os
import json

# Check if there is a settings-vector.json in place, and if so, load its values
if os.path.exists('settings-vector.json'):
    settingsFile = open('settings-vector.json')
    settingsFileData = json.load(settingsFile)
    for (key, value) in settingsFileData.items():
        if key == "text_prompts":
            text_prompts = {}
            for (k, v) in value.items():
                text_prompts[int(k)] = v
        else:
            setattr(settings, key, value)
    settingsFile.close()
print("Vector settings:")
print({k: v for k, v in vars(VectorSettings).items() if not k.startswith('_')} | vars(settings))


# 1. Set Up

In [ ]:
#@title 1.1 Folders, dependencies, diffvg, runtime devices
import subprocess, os, sys, platform, shutil

PROJECT_DIR = os.path.abspath(os.getcwd())
model_path = os.path.join(PROJECT_DIR, 'models')
outDirPath = os.path.join(PROJECT_DIR, 'images_out')
batchFolder = os.path.join(outDirPath, settings.batch_name)
os.makedirs(model_path, exist_ok=True)
os.makedirs(batchFolder, exist_ok=True)

def gitclone(url, targetdir=None):
    cmd = ['git', 'clone', url] + ([targetdir] if targetdir else [])
    print(subprocess.run(cmd, stdout=subprocess.PIPE).stdout.decode('utf-8'))

# Repo clones shared with the main notebook (skipped when already present)
if not os.path.exists(f'{PROJECT_DIR}/CLIP'):
    gitclone("https://github.com/openai/CLIP")
if not os.path.exists(f'{PROJECT_DIR}/guided-diffusion'):
    gitclone("https://github.com/crowsonkb/guided-diffusion")
if not os.path.exists(f'{PROJECT_DIR}/ResizeRight'):
    gitclone("https://github.com/assafshocher/ResizeRight.git")
for sub in ('CLIP', 'guided-diffusion', 'ResizeRight'):
    p = os.path.join(PROJECT_DIR, sub)
    if p not in sys.path:
        sys.path.append(p)

# --- diffvg: clone + patch + build if not already importable -----------------
# The 2020 upstream needs three fixes on a 2026 stack (verified July 2026,
# torch 2.13 / Python 3.12 / CUDA 12.0 / GCC 13 host):
#   1. pybind11 submodule -> v2.13.6 (bundled one predates Python 3.11 frames)
#   2. -std=c++11 -> c++17 (modern pybind11 + CUDA 12 thrust)
#   3. build with the newest gcc the CUDA toolkit accepts (nvcc 12.0 caps at 12)
#   4. setup.py's build_with_cuda probe (torch.cuda at *build* time) is
#      unreliable -> hard-force it on when nvcc exists
#   5. -static-libstdc++: conda-based Jupyter kernels resolve conda's OLD
#      libstdc++ (GLIBCXX_3.4.30 missing) even when the CLI works
def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs])

try:
    import pydiffvg
    print('pydiffvg already installed')
except ImportError:
    # pydiffvg's import chain needs these; install and retry BEFORE rebuilding —
    # a missing runtime dep (e.g. skimage) looks identical to a missing build
    _pip('scikit-image', 'cssutils', 'svgwrite', 'svgpathtools')
    try:
        import pydiffvg
        print('pydiffvg ok after installing runtime deps')
    except ImportError:
        pydiffvg = None
if pydiffvg is None:
    print('Building diffvg (one-time, a few minutes)...')
    dv = os.path.join(PROJECT_DIR, 'diffvg')
    if not os.path.exists(dv):
        print(subprocess.run(['git', 'clone', '--recursive', 'https://github.com/BachiLi/diffvg', dv],
                             stdout=subprocess.PIPE).stdout.decode('utf-8'))
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'v2.13.6'], cwd=f'{dv}/pybind11')
    subprocess.run(['git', 'checkout', 'FETCH_HEAD'], cwd=f'{dv}/pybind11')
    cml = open(f'{dv}/CMakeLists.txt').read().replace('-std=c++11', '-std=c++17').replace('CXX_STANDARD 11', 'CXX_STANDARD 17')
    open(f'{dv}/CMakeLists.txt', 'w').write(cml)
    have_nvcc = shutil.which('nvcc') is not None
    sp = open(f'{dv}/setup.py').read().replace('build_with_cuda = False',
                                               f'build_with_cuda = {have_nvcc}')
    open(f'{dv}/setup.py', 'w').write(sp)
    env = dict(os.environ)
    env['LDFLAGS'] = env.get('LDFLAGS', '') + ' -static-libstdc++'
    if have_nvcc:
        for g in ('gcc-12', 'gcc-11'):  # nvcc <=12.3 rejects gcc 13
            if shutil.which(g):
                env['CC'] = g
                env['CXX'] = g.replace('gcc', 'g++')
                break
    _pip('setuptools<81')
    r = subprocess.run([sys.executable, 'setup.py', 'install'], cwd=dv, env=env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout.decode('utf-8')[-2000:])
    import pydiffvg  # noqa: F401 - raises if the build failed

# --- Bezier Splatting GPU rasterizer (renderer 'splat'/'auto') -----------
# gsplat's CUDA extension must compile against torch's CUDA major (cu13).
# Without a system CUDA 13 toolkit, NVIDIA's pip wheels assemble one INSIDE
# the venv: nvidia-cuda-nvcc + nvidia-cuda-crt + nvidia-nvvm all install into
# site-packages/nvidia/cu13/ alongside torch's bundled headers/libs, forming a
# complete CUDA_HOME. (Verified July 2026; see DEVNOTES for the wheel-name maze.)
_want_splat = settings.renderer in ('auto', 'splat')
_have_splat = False
if _want_splat:
    try:
        from gsplat.project_gaussians_2d_scale_rot import project_gaussians_2d_scale_rot  # noqa
        _have_splat = True
        print('gsplat 2D kernels available')
    except ImportError:
        import torch as _t
        if _t.cuda.is_available() and int(_t.version.cuda.split('.')[0]) < 13:
            print(f"splat auto-build needs torch built for CUDA >= 13 (this env: cu{_t.version.cuda}); "
                  "using diffvg. For the fast renderer, switch the notebook kernel to one with "
                  "torch cu13+ (e.g. the 'pristine-bench' kernel).")
        elif _t.cuda.is_available():
            print('Building gsplat (one-time, a few minutes)...')
            _cu = _t.version.cuda.split('.')  # e.g. '13.0'
            _pin = f'{_cu[0]}.{_cu[1]}.*'
            _pip(f'nvidia-cuda-nvcc=={_pin}', f'nvidia-cuda-crt=={_pin}', f'nvidia-nvvm=={_pin}')
            import site
            _sp = site.getsitepackages()[0]
            _cuda_home = os.path.join(_sp, 'nvidia', f'cu{_cu[0]}')
            if not os.path.exists(os.path.join(_cuda_home, 'lib64')):
                os.symlink('lib', os.path.join(_cuda_home, 'lib64'))
            _gs = os.path.join(PROJECT_DIR, 'gsplat-2d')
            if not os.path.exists(_gs):
                subprocess.run(['git', 'clone', 'https://github.com/XingtongGe/gsplat', _gs])
                subprocess.run(['git', 'checkout', 'bcca3ecae966a052e3bf8dd1ff9910cf7b8f851d'], cwd=_gs)
            _env = dict(os.environ)
            _env['CUDA_HOME'] = _cuda_home
            _env['PATH'] = os.path.join(_cuda_home, 'bin') + os.pathsep + _env.get('PATH', '')
            _env['TORCH_CUDA_ARCH_LIST'] = f'{_t.cuda.get_device_capability()[0]}.{_t.cuda.get_device_capability()[1]}'
            _env['MAX_JOBS'] = '16'
            _r = subprocess.run([sys.executable, '-m', 'pip', 'install', '.', '--no-build-isolation'],
                                cwd=_gs, env=_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            print(_r.stdout.decode('utf-8')[-1500:])
            try:
                from gsplat.project_gaussians_2d_scale_rot import project_gaussians_2d_scale_rot  # noqa
                _have_splat = True
            except ImportError:
                print('gsplat build failed — falling back to diffvg renderer')
        else:
            print('No CUDA — splat renderer unavailable, using diffvg')
if settings.renderer == 'splat' and not _have_splat:
    raise RuntimeError("renderer 'splat' requested but gsplat is unavailable")
use_splat = _have_splat and settings.renderer in ('auto', 'splat')

import random, math, io, time, contextlib
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from IPython import display
from tqdm.notebook import tqdm
import pydiffvg

import clip
from resize_right import resize

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
device = DEVICE

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Rasterizer device: diffvg's GPU path allocates with cudaMallocManaged, which
# WSL2 emulates through host page faults -> ~30x slower than its CPU renderer
# (measured 8s vs 0.25s backward at 384px/96 paths, see DEVNOTES.md). CLIP and
# SDS always run on the GPU; only rasterization is affected.
is_wsl = 'microsoft' in platform.uname().release.lower()
if settings.render_gpu == 'on':
    render_on_gpu = True
elif settings.render_gpu == 'off':
    render_on_gpu = False
else:
    render_on_gpu = torch.cuda.is_available() and not is_wsl
pydiffvg.set_use_gpu(render_on_gpu)
render_device = torch.device('cuda:0') if render_on_gpu else torch.device('cpu')
if render_on_gpu:
    pydiffvg.set_device(render_device)
print(f'diffvg rasterizer on {render_device} (WSL2 detected: {is_wsl})')


In [ ]:
#@title 1.2 Guidance stack (shared math with the main notebook) + vector canvas

normalize = T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                        std=[0.26862954, 0.26130258, 0.27577711])

def spherical_dist_loss(x, y):
    x = F.normalize(x, dim=-1)
    y = F.normalize(y, dim=-1)
    return (x - y).norm(dim=-1).div(2).arcsin().pow(2).mul(2)

def parse_prompt(prompt):
    if prompt.startswith('http://') or prompt.startswith('https://'):
        vals = prompt.rsplit(':', 2)
        vals = [vals[0] + ':' + vals[1], *vals[2:]]
    else:
        vals = prompt.rsplit(':', 1)
    vals = vals + ['', '1'][len(vals):]
    return vals[0], float(vals[1])

def fetch(url_or_path):
    if str(url_or_path).startswith('http://') or str(url_or_path).startswith('https://'):
        import requests
        r = requests.get(url_or_path)
        r.raise_for_status()
        fd = io.BytesIO()
        fd.write(r.content)
        fd.seek(0)
        return fd
    return open(url_or_path, 'rb')

padargs = {}

class MakeCutoutsDango(nn.Module):
    def __init__(self, cut_size, Overview=4, InnerCrop=0, IC_Size_Pow=0.5, IC_Grey_P=0.2, skip_augs=False):
        super().__init__()
        self.cut_size = cut_size
        self.Overview = Overview
        self.InnerCrop = InnerCrop
        self.IC_Size_Pow = IC_Size_Pow
        self.IC_Grey_P = IC_Grey_P
        self.skip_augs = skip_augs
        self.augs = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.Lambda(lambda x: x + torch.randn_like(x) * 0.01),
            T.RandomAffine(degrees=10, translate=(0.05, 0.05), interpolation=T.InterpolationMode.BILINEAR),
            T.Lambda(lambda x: x + torch.randn_like(x) * 0.01),
            T.RandomGrayscale(p=0.1),
            T.Lambda(lambda x: x + torch.randn_like(x) * 0.01),
            T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        ])

    def forward(self, input):
        cutouts = []
        gray = T.Grayscale(3)
        sideY, sideX = input.shape[2:4]
        max_size = min(sideX, sideY)
        min_size = min(sideX, sideY, self.cut_size)
        output_shape = [1, 3, self.cut_size, self.cut_size]
        pad_input = F.pad(input, ((sideY-max_size)//2, (sideY-max_size)//2,
                                  (sideX-max_size)//2, (sideX-max_size)//2), **padargs)
        cutout = resize(pad_input, out_shape=output_shape)

        if self.Overview > 0:
            if self.Overview <= 4:
                if self.Overview >= 1: cutouts.append(cutout)
                if self.Overview >= 2: cutouts.append(gray(cutout))
                if self.Overview >= 3: cutouts.append(TF.hflip(cutout))
                if self.Overview == 4: cutouts.append(gray(TF.hflip(cutout)))
            else:
                for _ in range(self.Overview):
                    cutouts.append(cutout)

        if self.InnerCrop > 0:
            for i in range(self.InnerCrop):
                size = int(torch.rand([])**self.IC_Size_Pow * (max_size - min_size) + min_size)
                offsetx = torch.randint(0, sideX - size + 1, ())
                offsety = torch.randint(0, sideY - size + 1, ())
                cutout = input[:, :, offsety:offsety + size, offsetx:offsetx + size]
                if i <= int(self.IC_Grey_P * self.InnerCrop):
                    cutout = gray(cutout)
                cutout = resize(cutout, out_shape=output_shape)
                cutouts.append(cutout)

        cutouts = torch.cat(cutouts)
        if not self.skip_augs:
            cutouts = self.augs(cutouts)
        return cutouts

# ---- secondary diffusion model (verbatim from the main notebook, cell 1.6) --

def append_dims(x, n):
    return x[(Ellipsis, *(None,) * (n - x.ndim))]

def expand_to_planes(x, shape):
    return append_dims(x, len(shape)).repeat([1, 1, *shape[2:]])

def t_to_alpha_sigma(t):
    return torch.cos(t * math.pi / 2), torch.sin(t * math.pi / 2)

@dataclass
class DiffusionOutput:
    v: torch.Tensor
    pred: torch.Tensor
    eps: torch.Tensor

class ConvBlock(nn.Sequential):
    def __init__(self, c_in, c_out):
        super().__init__(nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(inplace=True))

class SkipBlock(nn.Module):
    def __init__(self, main, skip=None):
        super().__init__()
        self.main = nn.Sequential(*main)
        self.skip = skip if skip else nn.Identity()
    def forward(self, input):
        return torch.cat([self.main(input), self.skip(input)], dim=1)

class FourierFeatures(nn.Module):
    def __init__(self, in_features, out_features, std=1.):
        super().__init__()
        assert out_features % 2 == 0
        self.weight = nn.Parameter(torch.randn([out_features // 2, in_features]) * std)
    def forward(self, input):
        f = 2 * math.pi * input @ self.weight.T
        return torch.cat([f.cos(), f.sin()], dim=-1)

class SecondaryDiffusionImageNet2(nn.Module):
    def __init__(self):
        super().__init__()
        c = 64
        cs = [c, c * 2, c * 2, c * 4, c * 4, c * 8]
        self.timestep_embed = FourierFeatures(1, 16)
        self.down = nn.AvgPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.net = nn.Sequential(
            ConvBlock(3 + 16, cs[0]), ConvBlock(cs[0], cs[0]),
            SkipBlock([
                self.down, ConvBlock(cs[0], cs[1]), ConvBlock(cs[1], cs[1]),
                SkipBlock([
                    self.down, ConvBlock(cs[1], cs[2]), ConvBlock(cs[2], cs[2]),
                    SkipBlock([
                        self.down, ConvBlock(cs[2], cs[3]), ConvBlock(cs[3], cs[3]),
                        SkipBlock([
                            self.down, ConvBlock(cs[3], cs[4]), ConvBlock(cs[4], cs[4]),
                            SkipBlock([
                                self.down, ConvBlock(cs[4], cs[5]), ConvBlock(cs[5], cs[5]),
                                ConvBlock(cs[5], cs[5]), ConvBlock(cs[5], cs[4]), self.up,
                            ]),
                            ConvBlock(cs[4] * 2, cs[4]), ConvBlock(cs[4], cs[3]), self.up,
                        ]),
                        ConvBlock(cs[3] * 2, cs[3]), ConvBlock(cs[3], cs[2]), self.up,
                    ]),
                    ConvBlock(cs[2] * 2, cs[2]), ConvBlock(cs[2], cs[1]), self.up,
                ]),
                ConvBlock(cs[1] * 2, cs[1]), ConvBlock(cs[1], cs[0]), self.up,
            ]),
            ConvBlock(cs[0] * 2, cs[0]), nn.Conv2d(cs[0], 3, 3, padding=1),
        )
    def forward(self, input, t):
        timestep_embed = expand_to_planes(self.timestep_embed(t[:, None]), input.shape)
        v = self.net(torch.cat([input, timestep_embed], dim=1))
        alphas, sigmas = map(partial(append_dims, n=v.ndim), t_to_alpha_sigma(t))
        pred = input * alphas - v * sigmas
        eps = input * sigmas + v * alphas
        return DiffusionOutput(v, pred, eps)

# ---- vector canvas ----------------------------------------------------------

def blue_noise_centers(count, canvas_w, canvas_h, device, density=None, k=24):
    """Blue-noise shape placement via Mitchell's best-candidate algorithm
    (Mitchell, SIGGRAPH 1991) — the exact-count cousin of Poisson-disk
    sampling (Bridson 2007). i.i.d. uniform centers are a Poisson process:
    local density variance leaves holes and pileups that the optimizer must
    then repair through boundary-only gradients (slow). Best-candidate keeps
    the point spectrum blue — even coverage per shape — for an init-only cost.

    density (H,W nonnegative, optional): importance map. Candidates are drawn
    from it and min-distances are measured in units of the local target
    spacing (which shrinks as 1/sqrt(density)), giving importance-weighted
    blue noise — denser where the map says detail lives, still clump-free.
    A 5% uniform floor never starves a region."""
    wh = torch.tensor([float(canvas_w), float(canvas_h)], device=device)
    if density is not None:
        pm = density.detach().to(device).float().flatten().clamp_min(0)
        pm = pm + pm.mean() * 0.05 + 1e-12
        rel = pm / pm.mean()
    pts = torch.empty(count, 2, device=device)
    for i in range(count):
        kk = 1 if i == 0 else k
        if density is None:
            cand = torch.rand(kk, 2, device=device) * wh
            w = None
        else:
            idx = torch.multinomial(pm, kk, replacement=True)
            cand = torch.stack([(idx % canvas_w).float(), (idx // canvas_w).float()],
                               dim=1) + torch.rand(kk, 2, device=device)
            w = rel[idx].sqrt()
        if i == 0:
            pts[0] = cand[0]
            continue
        d = torch.cdist(cand, pts[:i]).amin(dim=1)
        pts[i] = cand[(d * w if w is not None else d).argmax()]
    return pts

def detail_density(img):
    """(1,3,H,W) image in [0,1] -> (H,W) edge-magnitude placement map:
    finite-difference gradient, softened with a 5x5 mean so it reads as a
    density rather than a hard edge mask (LIVE spawns components at
    high-error regions for the same reason)."""
    g = img.detach().mean(dim=1, keepdim=True)
    gx = F.pad(g[..., :, 1:] - g[..., :, :-1], (0, 1))
    gy = F.pad(g[..., 1:, :] - g[..., :-1, :], (0, 0, 0, 1))
    mag = (gx.pow(2) + gy.pow(2)).sqrt()
    return F.avg_pool2d(mag, 5, stride=1, padding=2)[0, 0]

def add_path_tier(shapes, groups, points_vars, color_vars,
                  count, radius_frac, num_segments, canvas_w, canvas_h,
                  canvas_snapshot=None, centers=None):
    """Append one tier of closed cubic blobs (VectorFusion-style init).

    Multi-scale, progressively-unlocked tiers are the vector analog of the
    raster notebook's overview->innercut arc: large shapes block in the
    composition, later smaller tiers carve detail (progressive-layer idea from
    LIVE, Ma et al. CVPR 2022, https://arxiv.org/abs/2206.04655). New tiers
    take their fill color from the canvas under them so they refine rather
    than restart. Path/color tensors live on the RENDER device; autograd
    carries gradients across the .to(DEVICE) in render_scene back into them.
    centers ((count,2) px, optional): precomputed centers — callers pass
    blue_noise_centers output (plain or importance-weighted) so tiers land
    with even, clump-free coverage; None falls back to uniform random."""
    for i in range(count):
        num_ctrl = torch.tensor([2] * num_segments, dtype=torch.int32)
        if centers is not None:
            center = centers[i].detach().cpu()
        else:
            center = torch.rand(2) * torch.tensor([float(canvas_w), float(canvas_h)])
        radius = (0.5 + torch.rand(()) * 0.5) * radius_frac * min(canvas_w, canvas_h)
        n_pts = 3 * num_segments
        angles = torch.linspace(0, 2 * math.pi, n_pts + 1)[:-1]
        pts = center + radius * torch.stack([angles.cos(), angles.sin()], dim=1)
        pts = pts + torch.randn_like(pts) * radius * 0.25
        pts = pts.to(render_device).contiguous().requires_grad_(True)
        path = pydiffvg.Path(num_control_points=num_ctrl, points=pts,
                             is_closed=True, stroke_width=torch.tensor(0.0))
        shapes.append(path)
        if canvas_snapshot is not None:
            cx = int(center[0].clamp(0, canvas_w - 1))
            cy = int(center[1].clamp(0, canvas_h - 1))
            rgb = canvas_snapshot[0, :, cy, cx].cpu() + torch.randn(3) * 0.05
            col = torch.cat([rgb.clamp(0, 1), 0.55 + 0.4 * torch.rand(1)])
        else:
            col = torch.cat([torch.rand(3), 0.55 + 0.4 * torch.rand(1)])
        col = col.to(render_device).requires_grad_(True)
        groups.append(pydiffvg.ShapeGroup(shape_ids=torch.tensor([len(shapes) - 1]),
                                          fill_color=col))
        points_vars.append(pts)
        color_vars.append(col)

def render_scene(shapes, groups, canvas_w, canvas_h, background, iteration):
    scene_args = pydiffvg.RenderFunction.serialize_scene(canvas_w, canvas_h, shapes, groups)
    img = pydiffvg.RenderFunction.apply(canvas_w, canvas_h, 2, 2, iteration, None, *scene_args)
    alpha = img[:, :, 3:4]
    img = alpha * img[:, :, :3] + (1 - alpha) * background.to(img.device)
    return img.permute(2, 0, 1).unsqueeze(0).to(DEVICE)  # 1,3,H,W in [0,1]

if use_splat:
    from bezier_splat_canvas import BezierSplatCanvas
    print('renderer: Bezier Splatting (GPU)')
else:
    print('renderer: diffvg')


# 2. Diffusion and CLIP model settings

In [ ]:
#@title Load the CLIP ensemble and the optional SDS prior
import hashlib, requests

clip_models = []
if settings.ViTB32: clip_models.append(clip.load('ViT-B/32', jit=False)[0].eval().requires_grad_(False).to(DEVICE))
if settings.ViTB16: clip_models.append(clip.load('ViT-B/16', jit=False)[0].eval().requires_grad_(False).to(DEVICE))
if settings.ViTL14: clip_models.append(clip.load('ViT-L/14', jit=False)[0].eval().requires_grad_(False).to(DEVICE))
if settings.RN50: clip_models.append(clip.load('RN50', jit=False)[0].eval().requires_grad_(False).to(DEVICE))
if settings.RN50x4: clip_models.append(clip.load('RN50x4', jit=False)[0].eval().requires_grad_(False).to(DEVICE))
if settings.ViTL14_laion2b or settings.ViTH14_laion2b:
    try:
        import open_clip
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'open_clip_torch'])
        import open_clip
    # fp16 with input-cast wrappers, mirroring the raster notebook's
    # _load_openclip_fp16 (open_clip does not auto-cast inputs)
    def _load_openclip_fp16(arch, tag):
        m = open_clip.create_model(arch, pretrained=tag).eval().requires_grad_(False)
        m = m.half().to(DEVICE)
        _enc_img = m.encode_image
        m.encode_image = (lambda x, _f=_enc_img: _f(x.half()))
        return m
    if settings.ViTL14_laion2b: clip_models.append(_load_openclip_fp16('ViT-L-14', 'laion2b_s32b_b82k'))
    if settings.ViTH14_laion2b: clip_models.append(_load_openclip_fp16('ViT-H-14', 'laion2b_s32b_b79k'))
assert clip_models, 'enable at least one CLIP model'
print(f'{len(clip_models)} CLIP model(s) loaded')
if settings.perf_compile:
    for _cm in clip_models:
        _cm.visual = torch.compile(_cm.visual)
    print(f'Perf: compiled {len(clip_models)} CLIP visual tower(s) (one-time warmup on first iterations)')

MODEL_SHAS = {
    'secondary_model_imagenet_2.pth': '983e3de6f95c88c81b2ca7ebb2c217933be1973b1ff058776b970f901584613a',
    '512x512_diffusion_uncond_finetune_008100.pt': '9c111ab89e214862b76e1fa6a1b3f1d329b1a88281885943d2cdbe357ad57648',
}
MODEL_URIS = {
    'secondary_model_imagenet_2.pth': 'https://huggingface.co/spaces/huggi/secondary_model_imagenet_2.pth/resolve/main/secondary_model_imagenet_2.pth',
    '512x512_diffusion_uncond_finetune_008100.pt': 'https://huggingface.co/lowlevelware/512x512_diffusion_unconditional_ImageNet/resolve/main/512x512_diffusion_uncond_finetune_008100.pt',
}

def ensure_model(fname):
    fpath = os.path.join(model_path, fname)
    if not os.path.exists(fpath):
        print(f'Downloading {fname}...')
        with requests.get(MODEL_URIS[fname], stream=True) as r:
            r.raise_for_status()
            with open(fpath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    with open(fpath, 'rb') as f:
        assert hashlib.sha256(f.read()).hexdigest() == MODEL_SHAS[fname], f'{fname} SHA mismatch'
    return fpath

# LAION aesthetic predictor (Schuhmann, improved-aesthetic-predictor):
# MLP over the L2-normalized OpenAI ViT-L/14 image embedding
aesthetic_head = None
aesthetic_clip = None
if settings.aesthetic_scale > 0:
    assert settings.ViTL14, 'aesthetic_scale needs the ViTL14 CLIP model enabled'
    # locate the ViT-L/14 model by its embed dim (768) and input resolution 224
    for _m in clip_models:
        try:
            if _m.visual.input_resolution == 224 and _m.visual.output_dim == 768:
                aesthetic_clip = _m
                break
        except AttributeError:
            continue
    assert aesthetic_clip is not None, 'could not locate ViT-L/14 in the ensemble'
    _apath = os.path.join(model_path, 'aesthetic_predictor_l14.pth')
    if not os.path.exists(_apath):
        import urllib.request
        urllib.request.urlretrieve(
            'https://github.com/christophschuhmann/improved-aesthetic-predictor/raw/main/sac%2Blogos%2Bava1-l14-linearMSE.pth', _apath)
    aesthetic_head = nn.Sequential(
        nn.Linear(768, 1024), nn.Dropout(0.2), nn.Linear(1024, 128),
        nn.Dropout(0.2), nn.Linear(128, 64), nn.Dropout(0.1),
        nn.Linear(64, 16), nn.Linear(16, 1))
    _sd = torch.load(_apath, map_location='cpu')
    aesthetic_head.load_state_dict({k.replace('layers.', ''): v for k, v in _sd.items()})
    aesthetic_head.eval().requires_grad_(False).to(DEVICE)
    print('aesthetic reward head loaded')

sds_model = None
sds_alphas_cumprod = None
if settings.sds_mode == 'secondary':
    sds_model = SecondaryDiffusionImageNet2()
    sds_model.load_state_dict(torch.load(ensure_model('secondary_model_imagenet_2.pth'), map_location='cpu'))
    sds_model.eval().requires_grad_(False).to(DEVICE)
    print('SDS prior: secondary model')
elif settings.sds_mode == 'primary':
    from guided_diffusion.script_util import create_model_and_diffusion, model_and_diffusion_defaults
    model_config = model_and_diffusion_defaults()
    model_config.update({
        'attention_resolutions': '32, 16, 8', 'class_cond': False,
        'diffusion_steps': 1000, 'rescale_timesteps': True,
        'timestep_respacing': '1000', 'image_size': 512, 'learn_sigma': True,
        'noise_schedule': 'linear', 'num_channels': 256, 'num_head_channels': 64,
        'num_res_blocks': 2, 'resblock_updown': True, 'use_checkpoint': False,
        'use_fp16': True, 'use_scale_shift_norm': True,
    })
    sds_model, _sds_diffusion = create_model_and_diffusion(**model_config)
    sds_model.load_state_dict(torch.load(ensure_model('512x512_diffusion_uncond_finetune_008100.pt'), map_location='cpu'))
    sds_model.requires_grad_(False).eval().to(DEVICE)
    sds_model.convert_to_fp16()
    if settings.perf_compile:
        sds_model = torch.compile(sds_model)
    sds_alphas_cumprod = torch.tensor(_sds_diffusion.alphas_cumprod, device=DEVICE, dtype=torch.float32)
    print('SDS prior: 512x512 guided-diffusion UNet' + (' (compiled)' if settings.perf_compile else ''))
else:
    print('SDS prior: none (pure CLIP guidance)')


# 3. Diffuse (as vectors)!

In [ ]:
#@title Run the vector optimization
side_x, side_y = settings.width_height

seed = settings.set_seed
if seed in ('random_seed', '', None):
    random.seed()
    seed = random.randint(0, 2**32)
seed = int(seed)
print(f'Using seed: {seed}')
torch.manual_seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)

def _as_sched(v):
    if isinstance(v, str):
        sched = eval(v)
        assert len(sched) == 1000, 'schedules must have 1000 entries'
        return sched
    return [v] * 1000

cut_overview = _as_sched(settings.cut_overview)
cut_innercut = _as_sched(settings.cut_innercut)
cut_ic_pow = _as_sched(settings.cut_ic_pow)
cut_icgray_p = _as_sched(settings.cut_icgray_p)

# target embeds, exactly as the main notebook's do_run builds model_stats
frame_prompt = text_prompts[0]
model_stats = []
for clip_model in clip_models:
    stat = {'clip_model': clip_model, 'target_embeds': [], 'weights': []}
    for prompt in frame_prompt:
        txt, weight = parse_prompt(prompt)
        emb = clip_model.encode_text(clip.tokenize(txt).to(DEVICE)).float()
        stat['target_embeds'].append(emb)
        stat['weights'].append(weight)
    stat['target_embeds'] = torch.cat(stat['target_embeds'])
    stat['weights'] = torch.tensor(stat['weights'], device=DEVICE)
    if stat['weights'].sum().abs() < 1e-3:
        raise RuntimeError('The weights must not sum to 0.')
    stat['weights'] /= stat['weights'].sum().abs()
    model_stats.append(stat)

# batch numbering, mirroring the main notebook's convention
batchNum = len([f for f in os.listdir(batchFolder) if f.endswith('_final.svg')])

tiers = settings.path_schedule or [[0.0, settings.num_paths, 0.08]]
tiers = [(float(f), int(c), float(r)) for f, c, r in tiers]
assert tiers[0][0] == 0.0, 'first path_schedule tier must start at 0'

init_target = None
lpips_model = None
if settings.init_image:
    _tgt = Image.open(fetch(settings.init_image)).convert('RGB').resize((side_x, side_y), Image.LANCZOS)
    init_target = TF.to_tensor(_tgt).unsqueeze(0).to(DEVICE)
    try:
        import lpips
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'lpips'])
        import lpips
    lpips_model = lpips.LPIPS(net='vgg', verbose=False).to(DEVICE)

def image_anchor_loss(img):
    # composite anchor, same spirit as the raster notebook's lpips init_scale:
    # MSE aligns color fields, LPIPS preserves structure/edges
    return F.mse_loss(img, init_target) +         0.15 * lpips_model(img * 2 - 1, init_target * 2 - 1).mean()

splat_canvas = None
shapes, groups, points_vars, color_vars = [], [], [], []
tier_counter = [0]

def register_splat_tier(new):
    groups_ = [{'params': [new['colors']], 'lr': settings.color_lr},
               {'params': [new['opacity']], 'lr': 0.01}]
    if new['use_gradient']:
        groups_.append({'params': [new['colors2']], 'lr': settings.color_lr})
        groups_.append({'params': [new['axis']], 'lr': 0.05})
    return groups_

def sample_tier_centers(count, density=None):
    # blue-noise placement for every tier (blue_noise_centers, cell 1.2):
    # even coverage per shape; with a density map, importance-weighted
    return blue_noise_centers(count, side_x, side_y, DEVICE, density=density)

# tier-1 placement: blue noise; with a warm-start target, weighted by its
# edge map so the first shape budget lands where the init has structure
_c0 = sample_tier_centers(tiers[0][1],
                          detail_density(init_target) if init_target is not None else None)

if use_splat:
    splat_canvas = BezierSplatCanvas(side_x, side_y, DEVICE)
    # with a warm-start target, tier-1 colors come from the target image
    _new = splat_canvas.add_tier(tiers[0][1], tiers[0][2],
                                 canvas_snapshot=init_target,
                                 centers_px=_c0,
                                 use_gradient=0 < settings.gradient_tiers)
    tier_counter[0] = 1
    # normalized coords (1 px = 2/min units); 3x because splat positional
    # gradients are weaker than diffvg's exact geometry gradients (measured)
    points_optim = torch.optim.Adam([_new['points']], lr=settings.points_lr * 3.0 * 2.0 / min(side_x, side_y))
    # opacity logits need a faster lr than [0,1] colors to keep up
    color_optim = torch.optim.Adam(register_splat_tier(_new))
else:
    add_path_tier(shapes, groups, points_vars, color_vars,
                  tiers[0][1], tiers[0][2], settings.num_segments, side_x, side_y,
                  canvas_snapshot=init_target.cpu() if init_target is not None else None,
                  centers=_c0)
    points_optim = torch.optim.Adam(points_vars, lr=settings.points_lr)
    color_optim = torch.optim.Adam(color_vars, lr=settings.color_lr)
pending_tiers = list(tiers[1:])

# learned palette anchors: k-means over the warm-start target (or initial
# shape colors), then optimized jointly with the shapes
palette = None
if settings.palette_scale > 0:
    with torch.no_grad():
        if init_target is not None:
            _pix = init_target[0].permute(1, 2, 0).reshape(-1, 3)
            _pix = _pix[torch.randperm(_pix.shape[0])[:8192]]
        else:
            _pix = torch.cat([c.detach().to(DEVICE) for c in
                              (splat_canvas.color_params if splat_canvas else color_vars)])
        _anchors = _pix[torch.randperm(_pix.shape[0])[:settings.palette_k]].clone()
        for _ in range(12):
            _d = (_pix[:, None, :] - _anchors[None]).pow(2).sum(-1)
            _assign = _d.argmin(1)
            for _k in range(settings.palette_k):
                _m = _assign == _k
                if _m.any():
                    _anchors[_k] = _pix[_m].mean(0)
    palette = _anchors.to(DEVICE).requires_grad_(True)
    color_optim.add_param_group({'params': [palette], 'lr': 0.005})

def render_current(bg, iteration=0):
    if splat_canvas is not None:
        return splat_canvas.render(bg.reshape(-1).to(DEVICE))
    return render_scene(shapes, groups, side_x, side_y, bg, iteration)

def unlock_tier(count, radius, snap):
    _density = None
    if init_target is not None:
        # error-guided densification: spawn detail where the render
        # deviates most from the warm-start target
        _density = (snap - init_target).abs().mean(dim=1)[0].detach()
    _centers = sample_tier_centers(count, _density)
    if splat_canvas is not None:
        _new = splat_canvas.add_tier(count, radius, canvas_snapshot=snap,
                                     centers_px=_centers,
                                     use_gradient=tier_counter[0] < settings.gradient_tiers)
        tier_counter[0] += 1
        points_optim.add_param_group({'params': [_new['points']]})
        for _g in register_splat_tier(_new):
            color_optim.add_param_group(_g)
    else:
        _n = len(points_vars)
        add_path_tier(shapes, groups, points_vars, color_vars,
                      count, radius, settings.num_segments, side_x, side_y,
                      canvas_snapshot=snap, centers=_centers)
        points_optim.add_param_group({'params': points_vars[_n:]})
        color_optim.add_param_group({'params': color_vars[_n:]})

bg_mode = settings.background
if bg_mode == 'auto':
    bg_mode = 'white'
if bg_mode == 'white':
    background = torch.ones(1, 1, 3)
elif bg_mode == 'black':
    background = torch.zeros(1, 1, 3)

loss_values = []
t_run0 = time.time()
preview_handle = None

# run header: prompt + config, shown above the progress bar for the whole run
_plines = '<br>'.join(
    f'&ldquo;<i>{parse_prompt(p)[0]}</i>&rdquo; <span style="opacity:.55">(weight {parse_prompt(p)[1]:g})</span>'
    for p in frame_prompt)
_cfg = (('Bézier Splatting (GPU)' if splat_canvas is not None else 'diffvg')
        + f' &middot; {side_x}&times;{side_y} &middot; {settings.iterations} iterations'
        + f' &middot; seed {seed}'
        + (f' &middot; warm-start: {os.path.basename(str(settings.init_image))}' if init_target is not None else ''))
display.display(display.HTML(
    f'<div style="font-family:sans-serif;line-height:1.6;margin:4px 0 8px 0">'
    f'<b>Rendering</b><br>{_plines}<br>'
    f'<span style="opacity:.55;font-size:.9em">{_cfg}</span></div>'))

pbar = tqdm(range(settings.iterations), desc='vector diffusion')
for i in pbar:
    while pending_tiers and i >= pending_tiers[0][0] * settings.iterations:
        _frac, _count, _radius = pending_tiers.pop(0)
        with torch.no_grad():
            snap = render_current(torch.ones(1, 1, 3))
        unlock_tier(_count, _radius, snap)
        _total = splat_canvas.num_shapes if splat_canvas is not None else len(shapes)
        print(f'iter {i}: unlocked tier of {_count} paths (radius {_radius}), total {_total}')
    points_optim.zero_grad(set_to_none=True)
    color_optim.zero_grad(set_to_none=True)
    if bg_mode == 'random':
        background = torch.rand(1, 1, 3)

    img = render_current(background, i)

    sched_i = min(999, int(1000 * i / settings.iterations))
    loss = torch.zeros((), device=DEVICE)

    # warm-start phase: pure image fit — the raster model composed the scene,
    # the vector scene learns to BE it before CLIP/SDS stylize it
    fit_iters = int(settings.init_fit_frac * settings.iterations) if init_target is not None else 0
    if i < fit_iters:
        loss = image_anchor_loss(img)
        loss.backward()
        if splat_canvas is not None:
            for group in list(points_optim.param_groups) + list(color_optim.param_groups):
                for prm in group['params']:
                    if prm.grad is not None:
                        torch.nan_to_num_(prm.grad, nan=0.0, posinf=0.0, neginf=0.0)
        points_optim.step(); color_optim.step()
        if splat_canvas is not None:
            splat_canvas.clamp_()
        else:
            with torch.no_grad():
                for col in color_vars: col.clamp_(0.0, 1.0)
        loss_values.append(float(loss.detach()))
        pbar.set_postfix(fit=f'{loss_values[-1]:.4f}')
        if (i + 1) % settings.display_rate == 0:
            with torch.no_grad():
                snap = render_current(torch.ones(1, 1, 3))
            pil = TF.to_pil_image(snap.squeeze(0).clamp(0, 1).cpu())
            if preview_handle is None:
                preview_handle = display.display(pil, display_id=True)
            else:
                preview_handle.update(pil)
        continue

    autocast_cm = torch.autocast('cuda', dtype=torch.float16) if settings.perf_autocast \
        else contextlib.nullcontext()
    for stat in model_stats:
        try:
            input_resolution = stat['clip_model'].visual.input_resolution
        except AttributeError:
            input_resolution = getattr(stat['clip_model'].visual, 'image_size', 224)
            if isinstance(input_resolution, (tuple, list)):
                input_resolution = input_resolution[0]
        cuts = MakeCutoutsDango(input_resolution,
                                Overview=cut_overview[sched_i],
                                InnerCrop=cut_innercut[sched_i],
                                IC_Size_Pow=cut_ic_pow[sched_i],
                                IC_Grey_P=cut_icgray_p[sched_i],
                                skip_augs=settings.skip_augs)
        for _ in range(settings.cutn_batches):
            with autocast_cm:
                clip_in = normalize(cuts(img))
                image_embeds = stat['clip_model'].encode_image(clip_in).float()
            if settings.quality_spherical_mean:
                # Karcher mean of cutout embeddings on the unit hypersphere
                # (same quality switch as the raster notebook; Crowson 2023,
                # https://github.com/crowsonkb/clip-guided-diffusion)
                emb = F.normalize(image_embeds, dim=-1)
                mu = F.normalize(emb.mean(0, keepdim=True), dim=-1)
                for _ka in range(4):
                    cosd = (emb * mu).sum(-1, keepdim=True).clamp(-1 + 1e-6, 1 - 1e-6)
                    ang = cosd.arccos()
                    tang = emb - mu * cosd
                    tang = tang / tang.norm(dim=-1, keepdim=True).clamp_min(1e-8) * ang
                    avg = tang.mean(0, keepdim=True)
                    nrm = avg.norm(dim=-1, keepdim=True).clamp_min(1e-8)
                    mu = F.normalize(mu * nrm.cos() + avg / nrm * nrm.sin(), dim=-1)
                dists = spherical_dist_loss(mu.unsqueeze(1), stat['target_embeds'].unsqueeze(0))
                loss = loss + dists.squeeze(0).mul(stat['weights']).sum() / settings.cutn_batches
            else:
                dists = spherical_dist_loss(image_embeds.unsqueeze(1),
                                            stat['target_embeds'].unsqueeze(0))
                dists = dists.view([cut_overview[sched_i] + cut_innercut[sched_i], 1, -1])
                loss = loss + dists.mul(stat['weights']).sum(2).mean(0).sum() / settings.cutn_batches

    if init_target is not None and settings.init_scale > 0:
        loss = loss + settings.init_scale * image_anchor_loss(img)

    if sds_model is not None:
        # Score Distillation Sampling (VectorFusion): the diffusion prior's
        # noise-prediction residual, injected as a gradient on the render.
        # The model forward runs no_grad — SDS skips the UNet Jacobian.
        x0 = img * 2 - 1
        t_lo, t_hi = settings.sds_t_range
        if settings.sds_anneal:
            # DreamTime-style sliding window: high noise (structure) early,
            # low noise (detail) late
            _prog = (i - fit_iters) / max(1, settings.iterations - fit_iters)
            # end at mid-noise: low-t SDS pulls toward the UNet's blurred
            # denoised mean and sands off late-run detail (measured)
            _c = 0.9 - (0.9 - 0.35) * _prog
            t_lo, t_hi = max(0.05, _c - 0.12), min(0.95, _c + 0.12)
        if settings.sds_mode == 'secondary':
            t = torch.rand(1, device=DEVICE) * (t_hi - t_lo) + t_lo
            alpha, sigma = t_to_alpha_sigma(t)
            eps = torch.randn_like(x0)
            x_t = alpha * x0 + sigma * eps
            with torch.no_grad():
                out = sds_model(x_t, t)
            sds_grad = (out.eps - eps).detach()
        else:  # primary
            x0 = F.interpolate(x0, (512, 512), mode='bilinear', align_corners=False)
            t = torch.randint(int(t_lo * 1000), max(int(t_lo * 1000) + 1, int(t_hi * 1000)), (1,), device=DEVICE)
            ac = sds_alphas_cumprod[t][:, None, None, None]
            eps = torch.randn_like(x0)
            x_t = ac.sqrt() * x0 + (1 - ac).sqrt() * eps
            with torch.no_grad():
                # UNet casts to its fp16 dtype internally (h = x.type(self.dtype))
                model_out = sds_model(x_t, t.float())[:, :3].float()
            sds_grad = (model_out - eps).detach()
        loss = loss + settings.sds_scale * (x0 * sds_grad).mean()

    # ---- quality rewards ----
    if settings.shape_reg_scale > 0 and splat_canvas is not None:
        _cpen, _apen = splat_canvas.shape_regularity_loss()
        loss = loss + settings.shape_reg_scale * (_cpen + 4.0 * _apen)
    if aesthetic_head is not None:
        _aest_in = normalize(F.interpolate(img, (224, 224), mode='bilinear', align_corners=False))
        _emb = aesthetic_clip.encode_image(_aest_in).float()
        loss = loss - settings.aesthetic_scale * aesthetic_head(F.normalize(_emb, dim=-1)).mean()
    if palette is not None:
        _plists = (splat_canvas.color_params +
                   [c2 for c2, g in zip(splat_canvas.color2_params, splat_canvas.tier_gradient) if g]
                   ) if splat_canvas else color_vars
        _cols = torch.cat([c if c.is_cuda else c.to(DEVICE) for c in _plists])
        loss = loss + settings.palette_scale *             (_cols[:, None, :] - palette[None]).pow(2).sum(-1).min(dim=1).values.mean()
    if settings.solidity_scale > 0 and splat_canvas is not None:
        _opas = torch.sigmoid(torch.cat(splat_canvas.opacity_params))
        loss = loss + settings.solidity_scale * (0.88 - _opas).relu().mean()

    loss.backward()
    if splat_canvas is not None:
        # sanitize grads BEFORE stepping: one inf/NaN grad (e.g. an
        # alpha-saturated splat) would otherwise poison Adam state and the
        # params permanently
        for group in list(points_optim.param_groups) + list(color_optim.param_groups):
            for prm in group['params']:
                if prm.grad is not None:
                    torch.nan_to_num_(prm.grad, nan=0.0, posinf=0.0, neginf=0.0)
    points_optim.step()
    color_optim.step()
    if splat_canvas is not None:
        splat_canvas.clamp_()
    else:
        with torch.no_grad():
            for col in color_vars:
                col.clamp_(0.0, 1.0)
            m = 0.1 * min(side_x, side_y)
            for pts in points_vars:
                pts[:, 0].clamp_(-m, side_x + m)
                pts[:, 1].clamp_(-m, side_y + m)

    loss_values.append(float(loss.detach()))
    pbar.set_postfix(loss=f'{loss_values[-1]:.4f}')

    last = i == settings.iterations - 1
    if (i + 1) % settings.display_rate == 0 or last:
        with torch.no_grad():
            snap = render_current(torch.ones(1, 1, 3))
        pil = TF.to_pil_image(snap.squeeze(0).clamp(0, 1).cpu())
        # update the preview in place via a display handle — clear_output
        # would also wipe the tqdm progress-bar widget (VS Code renders both
        # as cell output items)
        if preview_handle is None:
            preview_handle = display.display(pil, display_id=True)
        else:
            preview_handle.update(pil)
    if (i + 1) % settings.intermediate_saves == 0 and not last:
        with torch.no_grad():
            snap = render_current(torch.ones(1, 1, 3))
        TF.to_pil_image(snap.squeeze(0).clamp(0, 1).cpu()).save(
            f'{batchFolder}/{settings.batch_name}({batchNum})_{i+1:05}.png')
        if splat_canvas is not None:
            # intermediate SVGs: harvest the run at its best moment, not
            # only its last
            splat_canvas.to_svg(f'{batchFolder}/{settings.batch_name}({batchNum})_{i+1:05}.svg')

svg_path = f'{batchFolder}/{settings.batch_name}({batchNum})_final.svg'
scale = settings.export_png_size / max(side_x, side_y)
W2, H2 = int(side_x * scale), int(side_y * scale)
png_path = svg_path.replace('_final.svg', '_final.png')
if splat_canvas is not None:
    splat_canvas.to_svg(svg_path)
    big = splat_canvas.render_at(W2, H2)
    TF.to_pil_image(big.squeeze(0).clamp(0, 1).cpu()).save(png_path)
else:
    pydiffvg.save_svg(svg_path, side_x, side_y, shapes, groups)
    # high-res raster export: re-render the same scene scaled up (SVG is exact
    # at any size; this PNG is just a convenience copy)
    with torch.no_grad():
        big_shapes = []
        for s in shapes:
            big_shapes.append(pydiffvg.Path(num_control_points=s.num_control_points,
                                            points=s.points.detach() * scale,
                                            is_closed=True, stroke_width=torch.tensor(0.0)))
        scene_args = pydiffvg.RenderFunction.serialize_scene(W2, H2, big_shapes, groups)
        big = pydiffvg.RenderFunction.apply(W2, H2, 2, 2, 0, None, *scene_args)
        big = big[:, :, 3:4] * big[:, :, :3] + (1 - big[:, :, 3:4])
    TF.to_pil_image(big.permute(2, 0, 1).clamp(0, 1).cpu()).save(png_path)

print(f'\nDone in {time.time()-t_run0:.0f}s ({(time.time()-t_run0)/settings.iterations:.2f}s/it)')
print(f'SVG:  {svg_path}')
print(f'PNG:  {png_path}  ({W2}x{H2})')


### Notes

- **This is optimization, not sampling.** The classic notebook walks a noise tensor
  through the reverse diffusion process; here Adam walks Bézier control points and
  fill colors downhill on the same CLIP guidance losses (plus, optionally, a score
  distillation term from the same diffusion models). Same soul, different substrate.
- **The SVG is the artifact.** `_final.svg` opens in Inkscape/Illustrator/browsers,
  scales to any size, and every shape is individually editable. The PNG is a courtesy
  render.
- **WSL2 + diffvg**: the rasterizer runs on CPU under WSL2 (managed-memory
  emulation makes diffvg's GPU path ~30x slower there — measured). On native Linux,
  `render_gpu: 'auto'` will use the GPU. Either way CLIP/SDS run on the GPU.
- **Aesthetic expectations**: flat filled shapes — poster/icon/illustration
  territory. Disco's painterly grain is raster micro-texture and cannot survive
  vectorization by construction. Prompt suffixes like "flat vector art", "minimal
  poster", "flat illustration" work with the medium instead of against it.
- **Knob intuition**: `num_paths` is the detail budget; `iterations` the polish
  budget; overview cuts early -> innercut later (the default schedules) mirrors the
  classic composition-then-detail arc. `sds_scale` too high causes the photographic
  prior to fight the flat aesthetic (muddy, over-shaded shapes).
- Dev copy of the loop: `bench/vector_bench.py` (same code, argparse-driven, for
  headless A/B runs — see DEVNOTES verification recipe).
